# 1. Phân đoạn khách hàng với K-Means

**Mục tiêu:** Tìm số cụm tối ưu $k$, kiểm tra độ ổn định của K-Means, xuất bảng tinh chỉnh (`tuning_kmeans.csv`), cột nhãn `kmeans` trong `labels.csv`, và lưu mô hình `models/kmeans.joblib` cho app Streamlit.

## Lý thuyết cốt lõi
K-Means chia $n$ điểm dữ liệu thành $k$ cụm không giao nhau bằng cách tối thiểu hóa tổng bình phương khoảng cách nội cụm (**WCSS / Inertia**):

$$J = \sum_{j=1}^{k} \sum_{x_i \in C_j} \|x_i - \mu_j\|^2$$

Thuật toán lặp qua 2 bước:
1. **Gán cụm (Assignment):** Mỗi khách hàng được gán vào tâm $\mu_j$ gần nhất theo khoảng cách Euclidean.
2. **Cập nhật tâm (Update):** Tính lại tâm $\mu_j$ bằng trung bình cộng các điểm trong cụm $C_j$.

## Quyết định tiền xử lý & khởi tạo
- **Dùng dữ liệu đã `log1p + RobustScaler` từ TV1:** K-Means dùng trung bình và khoảng cách Euclidean nên rất nhạy với dữ liệu lệch phải (Monetary, Frequency) và ngoại biên.
- **Dùng `init='k-means++'` và `random_state=42`:** `k-means++` chọn các tâm ban đầu cách xa nhau để tránh kẹt cực tiểu địa phương; `random_state=42` giúp kết quả tái lập được trên máy của mọi thành viên.

In [ ]:
import sys
from pathlib import Path

# Thêm thư mục gốc của repo vào sys.path để import được clustering/models.py
# dù bạn mở notebook từ thư mục gốc hay từ thư mục clustering/
ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "clustering").exists() and (ROOT_DIR.parent / "clustering").exists():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_samples

from clustering.models import (
    RANDOM_STATE,
    fit_kmeans,
    load_scaled_data,
    run_grid_search,
    save_cluster_labels,
    save_model,
    save_tuning_results,
)

# Đọc dữ liệu chuẩn hóa của TV1 (tự tạo mock data nếu chưa có X_scaled.csv)
customer_ids, X_df, is_mock = load_scaled_data()
print(f"Kích thước dữ liệu đưa vào mô hình: {X_df.shape}")
X_df.head()

## Bước 1: Tinh chỉnh số cụm $k \in [2, 10]$
- **Vì sao dùng `ParameterGrid` thay vì `GridSearchCV`?** Bài toán gom cụm là học không giám sát (không có nhãn thật $y_{true}$), nên không chia fold chấm accuracy như bài toán phân loại. Thay vào đó, ta quét lưới $k$ và đánh giá đồng thời bằng 3 chỉ số nội bộ:
  - **Silhouette Score** ($\uparrow$ càng gần 1 càng tốt): đo độ gắn kết trong cụm so với độ tách biệt với cụm lân cận.
  - **Davies-Bouldin Index** ($\downarrow$ càng nhỏ càng tốt): tỷ lệ độ phân tán nội cụm trên khoảng cách giữa các tâm cụm.
  - **Calinski-Harabasz Index** ($\uparrow$ càng cao càng tốt): tỷ lệ phương sai giữa các cụm trên phương sai nội cụm.

In [ ]:
K_RANGE = list(range(2, 11))

# Quét k từ 2 đến 10 và lưu bảng kết quả tinh chỉnh
tuning_df = run_grid_search(
    X_df,
    model_builder=lambda n_clusters: KMeans(
        n_clusters=n_clusters,
        init="k-means++",
        n_init=10,
        random_state=RANDOM_STATE,
    ),
    param_grid={"n_clusters": K_RANGE},
)

save_tuning_results(tuning_df, "tuning_kmeans.csv")
tuning_df

## Bước 2: Biểu đồ Elbow & Silhouette theo $k$
- **Elbow Method:** Quan sát đường `Inertia` (giảm đơn điệu khi $k$ tăng) để tìm điểm "khủy tay" — nơi tốc độ giảm Inertia bắt đầu chững lại rõ rệt.
- **Kết hợp với bài toán:** Đề bài hướng tới khoảng 4 nhóm khách hàng đặc trưng (*VIP, Tiềm năng, Ngủ đông, Cần níu kéo*). Nếu Silhouette đạt đỉnh ở $k=2$ (thường gặp trên RFM thực tế) nhưng Elbow gợi ý $k=4$, ta cần cân nhắc cả độ tách biệt hình học lẫn ý nghĩa phân khúc kinh doanh.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Biểu đồ Elbow (Inertia)
axes[0].plot(tuning_df["n_clusters"], tuning_df["inertia"], marker="o")
axes[0].set_title("Elbow Method (Inertia theo k)")
axes[0].set_xlabel("Số cụm (k)")
axes[0].set_ylabel("Inertia (WCSS)")
axes[0].set_xticks(K_RANGE)
axes[0].grid(True, linestyle="--", alpha=0.5)

# Biểu đồ Silhouette trung bình
axes[1].plot(tuning_df["n_clusters"], tuning_df["silhouette"], marker="s", color="darkorange")
axes[1].set_title("Silhouette Score theo k")
axes[1].set_xlabel("Số cụm (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_xticks(K_RANGE)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Bước 3: Silhouette Plot chi tiết cho các giá trị $k$ ứng viên
Điểm Silhouette trung bình cao chưa đảm bảo mọi cụm đều tốt (có thể có 1 cụm rất lớn và 1 cụm toàn điểm âm). Vẽ phân bố Silhouette của từng điểm theo cụm giúp kiểm tra:
- Các cụm có độ dày (số lượng khách) cân đối hay bị lệch quá mức?
- Có cụm nào nhiều điểm âm hoặc đỉnh thấp hơn hẳn mức trung bình (đường đứt nét đỏ) không?

In [ ]:
# >>> CHỈNH 2-3 GIÁ TRỊ k TỐT NHẤT TỪ BIỂU ĐỒ TRÊN <<<
K_CANDIDATES = [3, 4, 5]


def plot_silhouette_candidates(X: pd.DataFrame, k_list: list[int]) -> None:
    '''Vẽ Silhouette plot cho từng giá trị k ứng viên để soi chất lượng từng cụm.'''
    fig, axes = plt.subplots(1, len(k_list), figsize=(5 * len(k_list), 4.5))
    if len(k_list) == 1:
        axes = [axes]

    X_arr = X.to_numpy()
    for ax, k in zip(axes, k_list):
        _, labels = fit_kmeans(X_arr, n_clusters=k)
        sample_vals = silhouette_samples(X_arr, labels)
        avg_score = float(np.mean(sample_vals))

        y_lower = 10
        for cluster_id in range(k):
            cluster_vals = np.sort(sample_vals[labels == cluster_id])
            y_upper = y_lower + len(cluster_vals)
            ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_vals, alpha=0.75)
            ax.text(-0.05, y_lower + 0.5 * len(cluster_vals), str(cluster_id))
            y_lower = y_upper + 10

        ax.axvline(avg_score, color="red", linestyle="--", label=f"Mean = {avg_score:.3f}")
        ax.set_title(f"Silhouette Plot (k = {k})")
        ax.set_xlabel("Silhouette coefficient")
        ax.set_ylabel("Cụm")
        ax.set_xlim(-0.2, 1.0)
        ax.set_yticks([])
        ax.legend(loc="lower right")

    plt.tight_layout()
    plt.show()


plot_silhouette_candidates(X_df, K_CANDIDATES)

## Bước 4: Kiểm tra độ ổn định với `init` và `n_init`
Thử so sánh giữa `init='k-means++'` với `init='random'` và thay đổi số lần khởi tạo lại `n_init`. Dùng **Adjusted Rand Index (ARI)** so sánh nhãn thu được với cấu hình chuẩn (`k-means++`, `n_init=10`); ARI gần `1.0` chứng tỏ phân cụm ổn định, ít phụ thuộc điểm khởi tạo.

In [ ]:
# >>> CHỈNH SỐ CỤM DỰ KIẾN ĐỂ KIỂM TRA ĐỘ ỔN ĐỊNH <<<
CHECK_K = 4

_, base_labels = fit_kmeans(X_df, n_clusters=CHECK_K, init="k-means++", n_init=10)

stability_rows = []
for init_method, n_init_val in [("k-means++", 5), ("k-means++", 10), ("k-means++", 20), ("random", 10)]:
    model, lbls = fit_kmeans(X_df, n_clusters=CHECK_K, init=init_method, n_init=n_init_val)
    ari = adjusted_rand_score(base_labels, lbls)
    sil = float(np.mean(silhouette_samples(X_df, lbls)))
    stability_rows.append(
        {
            "init": init_method,
            "n_init": n_init_val,
            "inertia": round(model.inertia_, 2),
            "silhouette": round(sil, 4),
            "ARI_vs_baseline": round(ari, 4),
        }
    )

pd.DataFrame(stability_rows)

## Bước 5: Chốt mô hình cuối & Lưu kết quả bàn giao
> **Ghi chú cho báo cáo Chương 4 (cập nhật sau khi chạy trên data thật):**
> - Số cụm lựa chọn `BEST_K = ...` vì: *(nêu điểm Elbow + giá trị Silhouette / Davies-Bouldin + độ ổn định ARI + tính hợp lý với 4 persona trong đề).*

In [ ]:
# >>> CHỈNH BEST_K CUỐI CÙNG SAU KHI XEM DỮ LIỆU THẬT <<<
BEST_K = 4
BEST_INIT = "k-means++"
BEST_N_INIT = 10

final_kmeans, kmeans_labels = fit_kmeans(
    X_df,
    n_clusters=BEST_K,
    init=BEST_INIT,
    n_init=BEST_N_INIT,
)

# Thống kê số lượng khách hàng ở từng cụm
counts = pd.Series(kmeans_labels, name="count").value_counts().sort_index()
print("Phân bố số khách hàng theo cụm (K-Means):")
print(counts)

# Lưu nhãn vào clustering/results/labels.csv và lưu model cho app Streamlit
save_cluster_labels(customer_ids, "kmeans", kmeans_labels)
save_model(final_kmeans, "kmeans.joblib")